# ML Assignment 2 — Model Training
Run the cells in order. This notebook downloads the built-in Breast Cancer Wisconsin Diagnostic dataset, trains five classifiers, and creates the required files.

In [ ]:
!pip -q install scikit-learn pandas numpy joblib

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
ROOT=Path('/content/ML_Classification_Assignment')
(ROOT/'model').mkdir(parents=True, exist_ok=True); (ROOT/'results').mkdir(exist_ok=True)
data=load_breast_cancer(as_frame=True); X,y=data.data,data.target
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
models={'Logistic Regression':Pipeline([('scaler',StandardScaler()),('classifier',LogisticRegression(max_iter=3000,random_state=42))]),'Decision Tree':Pipeline([('scaler',StandardScaler()),('classifier',DecisionTreeClassifier(max_depth=5,min_samples_leaf=3,random_state=42))]),'K-Nearest Neighbors':Pipeline([('scaler',StandardScaler()),('classifier',KNeighborsClassifier(n_neighbors=7))]),'Gaussian Naive Bayes':Pipeline([('scaler',StandardScaler()),('classifier',GaussianNB())]),'Support Vector Machine':Pipeline([('scaler',StandardScaler()),('classifier',SVC(kernel='rbf',C=1.0,probability=True,random_state=42))])}
files={'Logistic Regression':'logistic_regression.joblib','Decision Tree':'decision_tree.joblib','K-Nearest Neighbors':'k-nearest_neighbors.joblib','Gaussian Naive Bayes':'gaussian_naive_bayes.joblib','Support Vector Machine':'support_vector_machine.joblib'}
rows=[]
for name,m in models.items():
 m.fit(X_train,y_train); joblib.dump(m,ROOT/'model'/files[name]); p=m.predict(X_test); pp=m.predict_proba(X_test)[:,1]
 rows.append({'Model':name,'Accuracy':accuracy_score(y_test,p),'AUC':roc_auc_score(y_test,pp),'Precision':precision_score(y_test,p,zero_division=0),'Recall':recall_score(y_test,p,zero_division=0),'F1 Score':f1_score(y_test,p,zero_division=0),'MCC':matthews_corrcoef(y_test,p)})
pd.DataFrame(rows).to_csv(ROOT/'results'/'model_comparison.csv',index=False)
test=X_test.copy(); test['target']=y_test; test.to_csv(ROOT/'test_data.csv',index=False)
(ROOT/'feature_columns.json').write_text(json.dumps(list(X.columns),indent=2))
display(pd.DataFrame(rows).round(4))
print('Saved files to',ROOT)

After running this notebook, download the `ML_Classification_Assignment` folder from the Colab Files panel, or use the supplied ZIP project in this chat. The Streamlit `app.py` and requirements are included in the ZIP.